In [1]:
import sys
sys.path.append('..') 

from scripts.constants import *

import ee
import xee
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from dask.distributed import Client

In [2]:
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
ee.Initialize(project=GEE_PROJECT_NAME)
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')
client = Client(n_workers=2, threads_per_worker=2, memory_limit='16GB')

In [3]:
# # Define the region of interest (England boundaries)
# england_boundaries = gpd.read_file(VECTOR_IN_DIR / 'england_boundaries.shp')
# england_geometry = ee.Geometry.Polygon(england_boundaries.geometry.unary_union.exterior.coords)
# Define the region of interest (England boundaries) from Earth Engine
england_boundaries = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM1_NAME', 'England'))
england_geometry = england_boundaries.geometry()
# Define the date range for the year 2019
start_date = '2019-01-01'
end_date = '2019-12-31'
sentinel2_ee_path = "COPERNICUS/S2"

# Request Sentinel-2 data with low cloud coverage
sentinel2_collection = ee.ImageCollection(sentinel2_ee_path) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .filterBounds(england_geometry) \

# Print the number of images in the collection
print('Number of images in the collection:', sentinel2_collection.size().getInfo())

/maps/acz25/miniconda3/envs/3-30-300-env/lib/python3.10/site-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for COPERNICUS/S2! You are using a deprecated asset.
To ensure continued functionality, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2

  warnings.warn(warning, category=DeprecationWarning)


Number of images in the collection: 838


In [4]:
def calculate_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

# Add NDVI band to each image in the collection
sentinel2_with_ndvi = sentinel2_collection.map(calculate_ndvi)

# Function to calculate monthly NDVI
def monthly_ndvi(year, month):
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')
    monthly_collection = sentinel2_with_ndvi.filterDate(start_date, end_date)
    monthly_ndvi = monthly_collection.select('NDVI').mean()
    return monthly_ndvi.set('year', year).set('month', month)

# List to store monthly NDVI images
monthly_ndvi_images = []

# Loop through each month of 2019
for month in range(1, 13):
    monthly_ndvi_image = monthly_ndvi(2019, month)
    monthly_ndvi_images.append(monthly_ndvi_image)

# Print the list of monthly NDVI images
print(monthly_ndvi_images)
# Combine monthly NDVI images into a single image with separate bands
combined_ndvi = ee.Image.cat(monthly_ndvi_images).rename(['NDVI_Jan', 'NDVI_Feb', 'NDVI_Mar', 'NDVI_Apr', 'NDVI_May', 'NDVI_Jun', 'NDVI_Jul', 'NDVI_Aug', 'NDVI_Sep', 'NDVI_Oct', 'NDVI_Nov', 'NDVI_Dec'])

# Print the combined NDVI image
print(combined_ndvi.getInfo())

[<ee.image.Image object at 0x79c1a7b3d600>, <ee.image.Image object at 0x79c1a27b0190>, <ee.image.Image object at 0x79c1a27b0a30>, <ee.image.Image object at 0x79c1a27b1150>, <ee.image.Image object at 0x79c1a27b20e0>, <ee.image.Image object at 0x79c1a27b27a0>, <ee.image.Image object at 0x79c1a27b3040>, <ee.image.Image object at 0x79c1a27b38e0>, <ee.image.Image object at 0x79c1a27c01c0>, <ee.image.Image object at 0x79c1a27c0a60>, <ee.image.Image object at 0x79c1a27c1300>, <ee.image.Image object at 0x79c1a27c1e10>]
{'type': 'Image', 'bands': [{'id': 'NDVI_Jan', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'NDVI_Feb', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'NDVI_Mar', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0

In [5]:
import json
imd_lsoa_bua_boundaries_path = VECTOR_OUT_DIR / "IMD" / "English_IMD_2019_BUA_filtered_boundaries.geojson"
with open(imd_lsoa_bua_boundaries_path) as f:
    imd_lsoa_bua_boundaries_json = json.load(f)

imd_lsoa_bua_boundaries_ee = ee.FeatureCollection(imd_lsoa_bua_boundaries_json)

In [7]:
# Function to calculate mean NDVI for each feature
def calculate_mean_ndvi(feature):
    mean_dict = combined_ndvi.reduceRegions(
        collection=imd_lsoa_bua_boundaries_ee,
        reducer=ee.Reducer.mean()
    )
    return mean_dict

# Apply the function to calculate mean NDVI for each feature
mean_ndvi_features = calculate_mean_ndvi(imd_lsoa_bua_boundaries_ee)

# Get the results as a list of dictionaries
mean_ndvi_dict = mean_ndvi_features.getInfo()

# Print the results
print(mean_ndvi_dict)

EEException: Request payload size exceeds the limit: 10485760 bytes.